# 05. Explainability (primary feature set)

Displays outputs of `src/explainability.py`. Attributions describe **associations** learned from a cross-sectional survey; they are not causal effects.

In [1]:
import os, sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name != 'kavre_springs' and (PROJECT_ROOT / 'kavre_springs').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'kavre_springs'
elif PROJECT_ROOT.name != 'kavre_springs' and Path('..').resolve().name == 'kavre_springs':
    PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)


## 1. TreeSHAP (CatBoost)

In [3]:
display(pd.read_csv('results/metrics/feature_importance_shap.csv').head(15))
display(Image('results/figures/shap_feature_importance_bar.png'))
display(Image('results/figures/shap_summary_beeswarm.png'))

                 feature                   label  mean_abs_shap  rank
0              longitude               Longitude       0.412379     1
1               latitude                Latitude       0.224388     2
2              ndvi_mean                    NDVI       0.179296     3
3            elevation_m               Elevation       0.174049     4
4    Source Type_encoded    Source type (pond=1)       0.159551     5
5      precip_monsoon_mm         Monsoon precip.       0.144471     6
6              slope_deg                   Slope       0.138805     7
7    spring_density_500m  Spring density (500 m)       0.136565     8
8             aspect_sin       Aspect (eastness)       0.110844     9
9              ndwi_mean                    NDWI       0.102298    10
10  precip_dry_season_mm      Dry-season precip.       0.094199    11
11      precip_annual_mm          Annual precip.       0.088477    12
12            aspect_cos      Aspect (northness)       0.086215    13
13   landuse_agricul

## 2. Logistic regression coefficients with spatial-block bootstrap CIs
Logistic regression is the strongest model on the primary feature set, so its coefficients are the most defensible driver summary.

In [5]:
display(pd.read_csv('results/metrics/logistic_coefficients_block_bootstrap.csv').round(3))
display(Image('results/figures/logistic_coefficients.png'))

                  feature                   label  coef_per_sd  ci_low  ci_high  odds_ratio_per_sd  ci_excludes_zero
0        precip_annual_mm          Annual precip.       -1.289  -2.588    0.213              0.276             False
1       precip_monsoon_mm         Monsoon precip.        0.934  -0.417    2.174              2.545             False
2               ndvi_mean                    NDVI       -0.508  -0.797   -0.225              0.602              True
3               ndwi_mean                    NDWI       -0.412  -0.722   -0.084              0.663              True
4     landuse_agriculture         LU: agriculture       -0.398  -0.744   -0.056              0.671              True
5               longitude               Longitude        0.361   0.134    0.610              1.435              True
6    landuse_dense_forest        LU: dense forest       -0.335  -0.614   -0.065              0.716              True
7   landuse_sparse_forest       LU: sparse forest       -0.305  

## 3. Does ResGATv2 attention track terrain geometry?
Attention of models trained on training folds, evaluated on test receivers, compared with a placebo model trained on shuffled edge attributes. A physical reading is supported only where the real model's CI excludes the placebo's.

In [7]:
display(pd.read_csv('results/metrics/attention_analysis.csv').round(4))
display(Image('results/figures/attention_analysis.png'))

                    model                                          statistic  n_receivers    mean  ci_low  ci_high
0                    real   within-receiver Spearman(attention, edge length)         5396  0.2594  0.2459   0.2735
1                    real  attention(upslope senders) - attention(downslo...         4659 -0.0167 -0.0190  -0.0144
2  placebo_shuffled_edges   within-receiver Spearman(attention, edge length)         5396  0.0321  0.0202   0.0440
3  placebo_shuffled_edges  attention(upslope senders) - attention(downslo...         4659  0.0015 -0.0003   0.0033
<IPython.core.display.Image object>
